# AutoML Vision - Multiclass Classification
## Google Vertex AI AutoML Vision Alternative (Free)

Implementasi AutoML gratis menggunakan **AutoGluon** untuk klasifikasi multiclass:
- **Jenis pakaian**: Kaos (0) vs Hoodie (1)
- **Warna pakaian**: Merah (0), Kuning (1), Biru (2), Hitam (3), Putih (4)

### Keunggulan AutoGluon:
1. 🚀 State-of-the-art performance (comparable to Vertex AI)
2. 🎯 Automatic model selection & hyperparameter tuning
3. 🔥 Ensemble learning otomatis
4. 💰 Completely FREE & Open Source
5. 🎨 Transfer learning dari berbagai pretrained models
6. 📊 Built-in cross-validation & model stacking

## 1. Install Dependencies

In [ ]:
# Install AutoGluon for multimodal (vision) tasks
!pip install -q autogluon.multimodal torch torchvision
!pip install -q timm transformers accelerate

# Additional utilities
!pip install -q pillow opencv-python scikit-learn pandas numpy matplotlib seaborn

## 2. Import Libraries & Setup

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# AutoGluon imports
from autogluon.multimodal import MultiModalPredictor

# Set random seed for reproducibility
import random
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Setup plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Load & Explore Dataset

In [ ]:
# Define paths
TRAIN_DIR = 'train/train'
TEST_DIR = 'test/test'
TRAIN_CSV = 'train.csv'

# Load train data
train_df = pd.read_csv(TRAIN_CSV)

# Add image paths
train_df['image_path'] = train_df['id'].apply(
    lambda x: os.path.join(TRAIN_DIR, f"{x}.jpg") if os.path.exists(os.path.join(TRAIN_DIR, f"{x}.jpg")) 
    else os.path.join(TRAIN_DIR, f"{x}.png")
)

# Create label mappings
jenis_map = {0: 'Kaos', 1: 'Hoodie'}
warna_map = {0: 'Merah', 1: 'Kuning', 2: 'Biru', 3: 'Hitam', 4: 'Putih'}

# Add readable labels
train_df['jenis_name'] = train_df['jenis'].map(jenis_map)
train_df['warna_name'] = train_df['warna'].map(warna_map)

print("=" * 60)
print("📊 DATASET OVERVIEW")
print("=" * 60)
print(f"Total training images: {len(train_df)}")
print(f"\n{train_df.head(10)}")
print("\n" + "=" * 60)
print("📈 CLASS DISTRIBUTION")
print("=" * 60)
print("\n🔹 Jenis (Type):")
print(train_df['jenis_name'].value_counts())
print("\n🔹 Warna (Color):")
print(train_df['warna_name'].value_counts())

## 4. Visualize Dataset Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Jenis distribution
jenis_counts = train_df['jenis_name'].value_counts()
axes[0].bar(jenis_counts.index, jenis_counts.values, color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Distribusi Jenis Pakaian', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Jenis')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(jenis_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', va='bottom', fontweight='bold')

# Plot 2: Warna distribution
warna_counts = train_df['warna_name'].value_counts().sort_index()
colors = ['#FF4444', '#FFDD44', '#4488FF', '#222222', '#EEEEEE']
axes[1].bar(warna_counts.index, warna_counts.values, color=colors)
axes[1].set_title('Distribusi Warna Pakaian', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Warna')
axes[1].set_ylabel('Jumlah')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(warna_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', va='bottom', fontweight='bold')

# Plot 3: Cross tabulation
cross_tab = pd.crosstab(train_df['jenis_name'], train_df['warna_name'])
cross_tab.plot(kind='bar', ax=axes[2], color=colors, width=0.8)
axes[2].set_title('Distribusi Jenis × Warna', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Jenis')
axes[2].set_ylabel('Jumlah')
axes[2].legend(title='Warna', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("\n📊 Cross Tabulation (Jenis × Warna):")
print(cross_tab)

## 5. Visualize Sample Images

In [ ]:
# Sample images from each category
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Sample Images - 2 Jenis × 5 Warna', fontsize=16, fontweight='bold', y=1.02)

for i, jenis in enumerate([0, 1]):
    for j, warna in enumerate([0, 1, 2, 3, 4]):
        sample = train_df[(train_df['jenis'] == jenis) & (train_df['warna'] == warna)]
        if len(sample) > 0:
            img_path = sample.iloc[0]['image_path']
            img = Image.open(img_path)
            axes[i, j].imshow(img)
            axes[i, j].set_title(f"{jenis_map[jenis]} - {warna_map[warna]}", 
                                fontsize=11, fontweight='bold')
        else:
            axes[i, j].text(0.5, 0.5, 'No Data', ha='center', va='center', 
                           transform=axes[i, j].transAxes, fontsize=12)
        axes[i, j].axis('off')

plt.tight_layout()
plt.show()

## 6. Prepare Data for AutoGluon

AutoGluon membutuhkan format data khusus:
- DataFrame dengan kolom untuk image path dan label(s)
- Untuk multi-task learning, kita bisa train 2 model (jenis & warna) atau combined label

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare dataset for both tasks
# We'll create datasets for both jenis and warna classification

# Dataset for Jenis (Type) classification
df_jenis = train_df[['image_path', 'jenis']].copy()
df_jenis.columns = ['image', 'label']

# Dataset for Warna (Color) classification  
df_warna = train_df[['image_path', 'warna']].copy()
df_warna.columns = ['image', 'label']

# Split data: 80% train, 20% validation
train_jenis, val_jenis = train_test_split(df_jenis, test_size=0.2, random_state=42, stratify=df_jenis['label'])
train_warna, val_warna = train_test_split(df_warna, test_size=0.2, random_state=42, stratify=df_warna['label'])

print("=" * 60)
print("📦 DATASET SPLITS")
print("=" * 60)
print(f"\n🔹 JENIS (Type) Classification:")
print(f"  • Training set: {len(train_jenis)} images")
print(f"  • Validation set: {len(val_jenis)} images")
print(f"  • Classes: {sorted(df_jenis['label'].unique())}")

print(f"\n🔹 WARNA (Color) Classification:")
print(f"  • Training set: {len(train_warna)} images")
print(f"  • Validation set: {len(val_warna)} images")
print(f"  • Classes: {sorted(df_warna['label'].unique())}")

print("\n✅ Data preparation completed!")

## 7. AutoML Training - JENIS (Type) Classification

### 🤖 True AutoML Approach:
AutoGluon akan otomatis:
- ✅ Memilih arsitektur model terbaik (ResNet, ViT, Swin, EfficientNet, dll)
- ✅ Melakukan hyperparameter tuning
- ✅ Membuat ensemble dari multiple models
- ✅ Cross-validation otomatis

**Kita hanya perlu setting: `time_limit` dan `presets`!**

In [ ]:
import time

print("=" * 60)
print("🚀 AUTOML TRAINING - JENIS CLASSIFICATION")
print("=" * 60)
print("🤖 Fully Automated - AutoGluon will find the best model & pipeline!")
print("=" * 60)

# Initialize predictor for Jenis
predictor_jenis = MultiModalPredictor(
    label='label',
    problem_type='multiclass',
    eval_metric='accuracy',
    path='automl_jenis'
)

start_time = time.time()

# Train with FULL AutoML - Let AutoGluon decide everything!
# AutoGluon will automatically:
# 1. Try multiple model architectures (ResNet, ViT, Swin, EfficientNet, etc.)
# 2. Perform hyperparameter optimization
# 3. Create ensembles of best models
# 4. Select optimal preprocessing pipeline

predictor_jenis.fit(
    train_data=train_jenis,
    tuning_data=val_jenis,
    time_limit=600,  # 10 minutes - increase for better results (e.g., 3600 for 1 hour)
    presets='medium_quality',  # Options: 'best_quality', 'high_quality', 'medium_quality', 'low_quality'
    # NO hyperparameters specified - AutoGluon decides everything!
)

training_time = time.time() - start_time

print(f"\n✅ Training completed in {training_time/60:.2f} minutes!")
print(f"🎯 AutoGluon tried multiple models and selected the best one(s)!")
print("=" * 60)

## 8. Evaluate JENIS Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Evaluate on validation set
print("=" * 60)
print("📊 JENIS MODEL EVALUATION")
print("=" * 60)

# Get performance metrics
val_score = predictor_jenis.evaluate(val_jenis)
print(f"\n🎯 Validation Accuracy: {val_score['accuracy']:.4f}")

# Get predictions
predictions_jenis = predictor_jenis.predict(val_jenis)

# Classification report
print("\n📋 Classification Report:")
print(classification_report(val_jenis['label'], predictions_jenis, 
                          target_names=['Kaos', 'Hoodie']))

# Confusion matrix
cm = confusion_matrix(val_jenis['label'], predictions_jenis)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Kaos', 'Hoodie'],
            yticklabels=['Kaos', 'Hoodie'])
plt.title('Confusion Matrix - Jenis Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Feature importance (if available)
try:
    importance = predictor_jenis.feature_importance(val_jenis)
    print("\n📈 Feature Importance:")
    print(importance)
except:
    print("\n⚠️ Feature importance not available for this model")

## 8. Inspect AutoML Models Tried

AutoGluon mencoba berbagai model dan membuat leaderboard

In [ ]:
print("=" * 80)
print("🏆 JENIS MODEL LEADERBOARD - Models Tried by AutoGluon")
print("=" * 80)

# Get leaderboard of all models tried
leaderboard_jenis = predictor_jenis.leaderboard(train_jenis, silent=True)
print("\nTop Models for JENIS Classification:")
print(leaderboard_jenis.to_string())

print("\n" + "=" * 80)
print("📊 Best Model Info:")
print("=" * 80)
print(f"Best Model: {predictor_jenis.model_names()[0]}")
print(f"\nAll models tried: {predictor_jenis.model_names()}")

## 9. Evaluate JENIS Model

## 10. AutoML Training - WARNA (Color) Classification

In [ ]:
print("=" * 60)
print("🚀 AUTOML TRAINING - WARNA CLASSIFICATION")
print("=" * 60)
print("🤖 Fully Automated - AutoGluon will find the best model & pipeline!")
print("=" * 60)

# Initialize predictor for Warna
predictor_warna = MultiModalPredictor(
    label='label',
    problem_type='multiclass',
    eval_metric='accuracy',
    path='automl_warna'
)

start_time = time.time()

# Train with FULL AutoML - Let AutoGluon decide everything!
predictor_warna.fit(
    train_data=train_warna,
    tuning_data=val_warna,
    time_limit=600,  # 10 minutes - increase for better results
    presets='medium_quality',  # AutoGluon will optimize based on this preset
    # NO manual hyperparameters - fully automated!
)

training_time = time.time() - start_time

print(f"\n✅ Training completed in {training_time/60:.2f} minutes!")
print(f"🎯 AutoGluon tried multiple models and selected the best one(s)!")
print("=" * 60)

## 11. Inspect WARNA AutoML Models

In [ ]:
print("=" * 60)
print("📊 WARNA MODEL EVALUATION")
print("=" * 60)

# Get performance metrics
val_score_warna = predictor_warna.evaluate(val_warna)
print(f"\n🎯 Validation Accuracy: {val_score_warna['accuracy']:.4f}")

# Get predictions
predictions_warna = predictor_warna.predict(val_warna)

# Classification report
warna_names = ['Merah', 'Kuning', 'Biru', 'Hitam', 'Putih']
print("\n📋 Classification Report:")
print(classification_report(val_warna['label'], predictions_warna, 
                          target_names=warna_names))

# Confusion matrix
cm_warna = confusion_matrix(val_warna['label'], predictions_warna)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_warna, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=warna_names,
            yticklabels=warna_names)
plt.title('Confusion Matrix - Warna Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 80)
print("🏆 WARNA MODEL LEADERBOARD - Models Tried by AutoGluon")
print("=" * 80)

# Get leaderboard of all models tried
leaderboard_warna = predictor_warna.leaderboard(train_warna, silent=True)
print("\nTop Models for WARNA Classification:")
print(leaderboard_warna.to_string())

print("\n" + "=" * 80)
print("📊 Best Model Info:")
print("=" * 80)
print(f"Best Model: {predictor_warna.model_names()[0]}")
print(f"\nAll models tried: {predictor_warna.model_names()}")

## 12. Evaluate WARNA Model

## 13. Model Summary & Comparison

In [ ]:
# Create comprehensive summary
summary_data = {
    'Model': ['Jenis (Type)', 'Warna (Color)'],
    'Classes': [2, 5],
    'Train Samples': [len(train_jenis), len(train_warna)],
    'Val Samples': [len(val_jenis), len(val_warna)],
    'Accuracy': [val_score['accuracy'], val_score_warna['accuracy']],
}

summary_df = pd.DataFrame(summary_data)

print("=" * 80)
print("🏆 AUTOML MODEL SUMMARY")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

# Visualize performance comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
models = ['Jenis\n(Type)', 'Warna\n(Color)']
accuracies = [val_score['accuracy'], val_score_warna['accuracy']]
colors_bar = ['#FF6B6B', '#4ECDC4']

axes[0].bar(models, accuracies, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='90% Target')
axes[0].legend()
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 0.02, f'{v:.2%}', ha='center', fontweight='bold', fontsize=12)

# Dataset size comparison
x = np.arange(len(models))
width = 0.35
axes[1].bar(x - width/2, [len(train_jenis), len(train_warna)], width, 
           label='Train', color='#95E1D3', edgecolor='black')
axes[1].bar(x + width/2, [len(val_jenis), len(val_warna)], width,
           label='Validation', color='#F38181', edgecolor='black')
axes[1].set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
axes[1].set_title('Dataset Split', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n✅ Both AutoML models trained successfully!")
print(f"📦 Models saved to: 'automl_jenis/' and 'automl_warna/'")

## 14. Predict on Test Set